In [ ]:
from varphot import nbui as ui

ui.show(ui.header("07", "Kamera-Charakterisierung",
    "Kameraprofil (Gain, Rauschen, Linearitätsbereich, Offset) in `observer.json` und `PHOTOMETRIE_PY_DATA/`; NB00 liest es.",
    wann="**einmalig vor NB00**",
    blocks=[ui.para("**Aufnahmen bereitstellen** (je Gain-Modus und Temperatur, wie später bei den Messungen):"),
            ui.kv([("`bias/`", "≥ 6 Kurzbelichtungen, Deckel zu"),
                   ("`flats/`", "Paare gleicher Belichtung bei 8–12 Pegeln (5–80 % des Maximalwerts); stabile Lichtquelle"),
                   ("`linearity/`", "Belichtungsreihe 2 s, 4 s, 6 s … bis über die Sättigung, je 3×"),
                   ("`darks/`", "optional: ≥ 5 Belichtungszeiten, je 2 Bilder, Temperatur stabil")], 110),
            ui.note("ok", "Dann **Schritt 1** (Kamera wählen) bis **Schritt 7** (Ergebnis speichern) der Reihe nach ausführen. Der Demo-Modus (`MODE=demo`) läuft ohne eigene Bilder.")]))

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("07", "1.1", "Kamera und Einstellung wählen",
    intro="Die **Kamera-Vorlage** liefert, was vorab bekannt ist: Sensor, Pixelzahl, Bayer-Muster, Eingabeformat, Herstellerwerte zum Vergleich. Gemessen wird trotzdem selbst, **je Einstellung** (Gain-Modus bzw. ISO).",
    blocks=[ui.kv([("Prüfstatus", "**G** gelesen · **F** Testbericht · **E** Annahme · **?** unbekannt (siehe `varphot/camera_presets.py`)"),
                   ("Weitere Kameras", "Vorlage in `camera_presets.py` ergänzen oder „Eigene Kamera“ wählen"),
                   ("`MODE=\"demo\"`", "synthetische Bilder mit **bekannter Wahrheit** (zeigt, dass die Auswertung sie wiederfindet; es wird nichts gespeichert)"),
                   ("`MODE=\"real\"`", "eigene Bilder aus `BASE`"),
                   ("Umgebungsvariablen", "`VARPHOT_NB07_MODE=real`, `VARPHOT_CAMERA=lumix-s9`, `VARPHOT_CAMERA_SETTING=\"ISO 100\"`")], 150)]))

In [ ]:
import os, json, datetime, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from varphot import (pair_stats, gain_from_pair, ptc_fit, read_noise_e, full_well_e, linearity_curve, dark_current_e_per_s,
                     detect_saturation, exposure_from_header, iso_from_header, get_preset, preset_choices)
from varphot.camera import PLANES, summarize_frame, group_by_exposure, _clipped_mean
from varphot.camera_presets import slug
from varphot.observer import Camera, camera_label, load_config, save_config, upsert_camera, data_dir
from varphot.batch import read_frame
from varphot.synthetic import bayer_frame
from varphot.testdata import testdata_dir

MODE = os.environ.get("VARPHOT_NB07_MODE", "demo")          # "demo" oder "real"
w_cam = widgets.Dropdown(options=preset_choices(), value=os.environ.get("VARPHOT_CAMERA", "ares-c-pro"), description="Kamera:",
                         style={"description_width": "90px"}, layout=widgets.Layout(width="520px"))
w_set = widgets.Dropdown(description="Einstellung:", style={"description_width": "90px"}, layout=widgets.Layout(width="520px"))
def _upd(*_):
    opts = list(get_preset(w_cam.value).settings)
    w_set.options = opts
    want = os.environ.get("VARPHOT_CAMERA_SETTING")
    w_set.value = want if want in opts else opts[0]
w_cam.observe(_upd, names="value"); _upd()
display(widgets.VBox([w_cam, w_set]))

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("07", "1.2", "Gewählte Kamera und Einstellung übernehmen",
    intro="Die Zelle direkt darunter ausführen, **nachdem** Kamera und Einstellung in Schritt 1.1 gewählt wurden. Sie zeigt die Vorlage samt Hinweisen. Nur bei „Eigene Kamera“ die Werte `MODEL`, `PATTERN`, `BITS` in dieser Zelle eintragen (leeres Bayer-Muster = Monochrom, `BITS=None` = unbekannt, wird aus dem Maximalwert abgeleitet)."))

In [ ]:
from varphot import nbui as ui
PRESET, SETTING = get_preset(w_cam.value), w_set.value
VENDOR_V = PRESET.settings[SETTING]
MODEL, PATTERN, BITS, ROI = PRESET.model or "Meine Kamera", PRESET.bayer_pattern or "RGGB", PRESET.bit_depth, 100
if PRESET.key == "eigene":                  # <- hier eintragen
    MODEL, PATTERN, BITS = "Meine Kamera", "RGGB", None
BASE = Path(os.environ.get("VARPHOT_CAMERA_DIR") or testdata_dir() / "camera" / f"{slug(MODEL)}-{slug(SETTING)}")
DIRS = {k: BASE / k for k in ("bias", "flats", "linearity", "darks")}
NOMINAL = (2 ** BITS - 1) if BITS else None
ui.say(f"{MODEL} — {SETTING}")
ui.say(f"Sensor: {PRESET.sensor}")
ui.say(f"Ausgabe {PRESET.format_px[0]} x {PRESET.format_px[1]} px, Pixel {PRESET.pixel_um} µm [{PRESET.pixel_status}], Bayer {PATTERN} [{PRESET.bayer_status}], "
      f"Bittiefe {BITS or 'unbekannt'} [{PRESET.bit_status}], Eingabe: {PRESET.raw_input}, AAVSO-OBSTYPE (vorläufig): {PRESET.obstype}")
ui.say(f"Herstellerwerte [{VENDOR_V.status}]: Gain {VENDOR_V.gain_e_per_adu}, Rauschen {VENDOR_V.read_noise_e}, Full Well {VENDOR_V.full_well_e} — {VENDOR_V.source}")
for n in PRESET.notes:
    ui.say(" •", n)
if PRESET.raw_input == "raw":
    ui.say("\nRAW-Eingabe: Bilder zuerst mit Siril nach FITS wandeln, OHNE zu debayern (CFA bleibt erhalten):\n"
          "  siril -d <Ordner_mit_RAW> -s tools/siril_convert_raw.ssf   (oder in Siril: convertraw <name> -out=./fits)\n"
          "Danach EXPTIME/ISO in der FITS-Kopfzeile prüfen (Siril-Doku nennt die Übernahme nicht).")
VENDOR = dict(gain=VENDOR_V.gain_e_per_adu, read_noise=VENDOR_V.read_noise_e, full_well=VENDOR_V.full_well_e)

if MODE == "demo":
    TRUTH = dict(gain=4.4, read_noise=4.5, offset=100.0, max_adu=2 ** (BITS or 14) - 1, dark=0.5)   # was die Auswertung wiederfinden soll
    ui.say("\nDEMO-Modus: synthetische Bilder, Wahrheit =", TRUTH)
else:
    ui.say("\nREAL-Modus: Bilder aus", BASE)

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("07", 2, "Bilder laden und verdichten", intro="Je Bild bleiben nur der Mittelbereich jeder Farbebene, der Maximalwert und der Median übrig. Das spart Speicher bei großen Bildern."))

In [ ]:
RAW_EXT = {".rw2", ".cr2", ".cr3", ".nef", ".arw", ".dng", ".orf", ".raf"}
def _exptime(hdr, name):
    t = exposure_from_header(hdr)
    if t is None:                                   # Notnagel: "..._2.5s..." im Dateinamen
        m = re.search(r"(\d+(?:\.\d+)?)\s*s(?:[_.\-]|$)", name)
        t = float(m.group(1)) if m else None
    if t is None:
        raise RuntimeError(f"{name}: keine Belichtungszeit in der FITS-Kopfzeile (EXPTIME/EXPOSURE) und keiner im Dateinamen — beim Wandeln mitgeben")
    return t

def load_dir(folder):
    folder = Path(folder)
    files = sorted(list(folder.glob("*.fit*")))
    if not files and any(f.suffix.lower() in RAW_EXT for f in folder.iterdir()):
        raise RuntimeError(f"{folder}: nur RAW-Dateien — zuerst mit Siril nach FITS wandeln (ohne Debayern), siehe Schritt 1")
    recs = []
    for p in files:
        img, hdr, _ = read_frame(p)
        r = summarize_frame(img, PATTERN, ROI)
        r["exptime"], r["file"], r["iso"] = _exptime(hdr, p.name), p.name, iso_from_header(hdr)
        recs.append(r)
    return recs

def synth(signal_e=0.0, exptime=0.001, dark_e=0.0):
    img = bayer_frame((400, 600), signal_e, TRUTH["gain"], TRUTH["read_noise"], TRUTH["offset"], TRUTH["max_adu"], PATTERN,
                      dark_e=dark_e, rng=RNG)
    if os.environ.get("VARPHOT_NB07_DEMO_COMPRESS"):                                  # zeigt die Erkennung: ab 2000 ADU nur jeder 4. Wert
        img = np.where(img > 2000, 2000 + np.round((img - 2000) / 4) * 4, img)
    r = summarize_frame(img, PATTERN, ROI); r["exptime"] = exptime; r["file"] = "synthetisch"
    return r

if MODE == "demo":
    RNG = np.random.default_rng(7)
    BIAS = [synth() for _ in range(6)]
    levels = [1500, 4000, 9000, 18000, 30000, 42000, 54000, 64000]                     # Elektronen je Pixel
    FLATS = [synth(s, exptime=s / 1000.0) for s in levels for _ in range(2)]
    LIN = [synth(2300.0 * t, exptime=t) for t in range(2, 42, 2) for _ in range(3)]    # sättigt bei ~31 s
    DARKS = [synth(0.0, exptime=t, dark_e=TRUTH["dark"] * t) for t in (10, 30, 60, 120, 240) for _ in range(2)]
else:
    BIAS, FLATS, LIN = (load_dir(DIRS[k]) for k in ("bias", "flats", "linearity"))
    DARKS = load_dir(DIRS["darks"]) if DIRS["darks"].is_dir() else []
ui.say(f"Bias {len(BIAS)}, Flats {len(FLATS)}, Linearität {len(LIN)}, Darks {len(DARKS)} Bilder")

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("07", "3.1", "Offset und Maximalwert",
    blocks=[ui.kv([("Offset", "Median der Bias-Pegel"),
                   ("Maximalwert", "hellster Pixel der gesättigten Aufnahmen (Linearitätsreihe, Flats). Wirklich gesättigt ist er, wenn mehrere Bilder **genau denselben** höchsten Wert zeigen"),
                   ("Unbekannte Bittiefe", "(z. B. Lumix S9) ist das die einzige Quelle: ohne Sättigung bricht das Notebook ab, statt einen falschen Wert zu verwenden; die Bittiefe wird dann aus dem Maximalwert abgeleitet")])]))

In [ ]:
offset = {p: float(np.median([_clipped_mean(r["rois"][p]) for r in BIAS])) for p in PLANES}
SAT = detect_saturation([r["max"] for r in LIN + FLATS], nominal=NOMINAL)
if SAT["saturated"]:
    MAX_ADU = SAT["max_adu"]
elif NOMINAL:
    MAX_ADU = NOMINAL
    ui.say(f"WARNUNG: nicht gesättigt (höchster Wert {SAT['max_adu']:.0f}); nominelle Grenze {NOMINAL} verwendet — Full Well und Linearitätsgrenze sind dann nur Annahmen")
else:
    raise RuntimeError("Bittiefe unbekannt und keine gesättigte Aufnahme: bitte die Linearitätsreihe bis über die Sättigung verlängern (mindestens 3 Bilder am Anschlag)")
BITS_USED = BITS or SAT["bits"]
ui.say("Offset je Ebene [ADU]:", {k: round(v, 1) for k, v in offset.items()})
ui.say(f"Maximalwert {MAX_ADU:.0f} ADU (gesättigt: {SAT['saturated']}, {SAT['n_at_max']} Bilder am Anschlag) → Bittiefe {BITS_USED}"
      + ("" if not BITS else f" (Vorlage: {BITS})"))

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("07", "3.2", "RAW-Kompression prüfen (Histogrammlücken)",
    intro="Manche Kameras speichern RAW **verlustbehaftet** (bei hellen Pixeln nur jeder 2., 4. … Wert) oder skalieren 12/14 Bit auf 16 Bit. Dann fehlen ganze ADU-Werte, und Gain und Ausleserauschen werden verfälscht. Deshalb **vor** der Gain-Messung prüfen.",
    blocks=[ui.kv([("Prüfung", "je Flat-Aufnahme und Farbebene wird im Kern der Verteilung (Median ± 2σ) gezählt, wie viele ganzzahlige Werte fehlen, obwohl sie bei dem vorhandenen Rauschen vorkommen müssten. Unkomprimiert: ≈ 0 % Lücken, Teiler 1"),
                   ("Grenze", "braucht Pegel mit genug Rauschen (σ ≥ ~2 ADU); sehr dunkle Flats sind nicht beurteilbar"),
                   ("Urteil", "Verdacht, wenn bei mindestens zwei Pegeln > 5 % Lücken oder ein Teiler > 1 auftritt (Startwert)")])]))

In [ ]:
from varphot import histogram_gaps, compression_verdict
rows, by_plane = [], {p: [] for p in PLANES}
for r in FLATS + LIN:
    for p in PLANES:
        h = histogram_gaps(r["rois"][p])
        by_plane[p].append(h)
        rows.append(dict(Ebene=p, Pegel_ADU=round(h["median"]), sigma=round(h["sigma"], 1), geprüft=h["n_tested"], Lücken_pct=round(100 * h["gap_fraction"], 1) if h["reliable"] else np.nan,
                         Teiler=h["step"], verlässlich=h["reliable"]))
GAPS = pd.DataFrame(rows).drop_duplicates(["Ebene", "Pegel_ADU"]).sort_values(["Ebene", "Pegel_ADU"])
verdicts = {p: compression_verdict(by_plane[p]) for p in PLANES}
for p, v in verdicts.items():
    ui.say(f"{p:>2}: {v['reason']}")
COMPRESSED = any(v["suspect"] for v in verdicts.values())
if COMPRESSED:
    ui.say("\n⚠ Verdacht auf komprimierte/skalierte RAW-Daten: Gain/Rauschen unten NICHT verwenden, bis das geklärt ist "
          "(Kamera auf unkomprimiertes RAW umstellen, Siril-Wandlung/Skalierung prüfen).")
ui.show(ui.table(GAPS[GAPS["verlässlich"]].groupby("Ebene").tail(4).round(1).set_index(["Ebene", "Pegel_ADU"]).drop(columns="verlässlich")))

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("07", 4, "Gain (Photon-Transfer) je Farbebene",
    blocks=[ui.kv([("Messung", "je Flat-Paar Mittelwert und zeitliche Varianz (die Differenz der zwei Bilder entfernt Flatfield-Struktur). Gerade `Varianz = Signal/Gain + const` über alle Pegel; Gain = 1/Steigung"),
                   ("Gegenprobe", "Einzelpaar-Formel des Handbuchs (Anhang A) beim Paar nahe 2/3 des Maximalwerts. Die Ebenen sollten denselben Gain liefern")])]))

In [ ]:
pairs = group_by_exposure(FLATS)
rows, ptc_data = [], {}
for p in PLANES:
    means, varis = [], []
    for t, grp in pairs.items():
        m, v = pair_stats(grp[0]["rois"][p], grp[1]["rois"][p], ROI)
        means.append(m); varis.append(v)
    fit = ptc_fit(means, varis, offset[p], MAX_ADU)
    # Einzelpaar nahe 2/3 des Maximalwerts + Bias-Paar
    k = int(np.argmin([abs(m - offset[p] - 0.66 * (MAX_ADU - offset[p])) for m in means]))
    grp = list(pairs.values())[k]
    g_pair = gain_from_pair(grp[0]["rois"][p], grp[1]["rois"][p], BIAS[0]["rois"][p], BIAS[1]["rois"][p], ROI)
    rn = read_noise_e(BIAS[0]["rois"][p], BIAS[1]["rois"][p], fit["gain"], ROI)
    rows.append(dict(Ebene=p, Offset_ADU=offset[p], Gain_PTC=fit["gain"], Gain_Paar=g_pair, R2=fit["r2"], n_Paare=fit["n"],
                     Ausleserauschen_e=rn, FullWell_e=full_well_e(fit["gain"], MAX_ADU)))
    ptc_data[p] = (np.array(means) - offset[p], np.array(varis), fit)
TAB = pd.DataFrame(rows).set_index("Ebene")
ui.show(ui.table(TAB.round(3)))
spread = (TAB["Gain_PTC"].max() - TAB["Gain_PTC"].min()) / TAB["Gain_PTC"].mean() * 100
ui.say(f"Streuung des Gains über die Ebenen: {spread:.1f} % (sollte klein sein; > 5 % → Pegel/Lichtquelle/Muster prüfen)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for p, (s, v, fit) in ptc_data.items():
    ax[0].plot(s, v, "o", ms=4, label=p)
    xs = np.linspace(0, s.max(), 20); ax[0].plot(xs, fit["slope"] * xs + fit["intercept"], lw=0.8)
ax[0].set_xlabel("Signal − Offset [ADU]"); ax[0].set_ylabel("Varianz [ADU²]"); ax[0].set_title("Photon-Transfer-Kurve (Steigung = 1/Gain)"); ax[0].legend()
ax[1].bar(TAB.index, TAB["Gain_PTC"], color="C0", label="PTC"); ax[1].plot(TAB.index, TAB["Gain_Paar"], "ko", label="Einzelpaar")
ax[1].set_ylabel("Gain [e⁻/ADU]"); ax[1].set_title("Gain je Ebene"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("07", 5, "Linearität",
    blocks=[ui.kv([("Messung", "Mittelwert eines festen Bereichs (Grünebene G1) gegen die Belichtungszeit, gemittelt über die drei Bilder je Belichtung. Zählrate `(ADU − a)/t` gegen eine Referenz-Gerade"),
                   ("Linear", "heißt hier |Abweichung| ≤ 1 % (Startwert). Das ergibt die ADU-Grenzen, innerhalb derer Sterne liegen dürfen"),
                   ("Auffällig", "Abweichungen bei kurzen Zeiten deuten auf reale Verschlusszeit oder Rolling Shutter")])]))

In [ ]:
lin_groups = {t: g for t, g in sorted({}.items())}
by_t = {}
for r in LIN:
    by_t.setdefault(round(r["exptime"], 3), []).append(r)
ts = np.array(sorted(by_t))
mean_adu = np.array([np.mean([_clipped_mean(x["rois"]["G1"]) for x in by_t[t]]) for t in ts])
spread_t = np.array([np.std([_clipped_mean(x["rois"]["G1"]) for x in by_t[t]]) / max(np.mean([_clipped_mean(x["rois"]["G1"]) for x in by_t[t]]), 1) * 100 for t in ts])
if spread_t.max() > 0.5:
    ui.say(f"WARNUNG: Lichtquelle instabil? Streuung der drei Bilder bis {spread_t.max():.2f} % — Linearitätstest dann nicht gültig (Handbuch Anhang E)")
LINRES = linearity_curve(ts, mean_adu, max_adu=MAX_ADU, tol_pct=1.0)
ui.say(f"Offset (Achsenabschnitt) {LINRES['a']:.1f} ADU, Zählrate {LINRES['b']:.1f} ADU/s")
ui.say(f"linear (±1 %) von {LINRES['t_min']:.0f} bis {LINRES['t_max']:.0f} s = {LINRES['adu_min']:.0f} bis {LINRES['adu_max']:.0f} ADU "
      f"({100 * LINRES['adu_max'] / MAX_ADU:.0f} % des Maximalwerts)")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ts, mean_adu, "o-", ms=4); ax[0].axhline(MAX_ADU, color="r", ls=":", label="Maximalwert"); ax[0].set_xlabel("Belichtung [s]"); ax[0].set_ylabel("Mittelwert G1 [ADU]"); ax[0].legend()
ax[1].plot(ts, LINRES["residual_pct"], "o-", ms=4); ax[1].axhspan(-1, 1, color="g", alpha=0.15); ax[1].set_ylim(-10, 10)
ax[1].set_xlabel("Belichtung [s]"); ax[1].set_ylabel("Zählratenabweichung [%]"); ax[1].set_title("grün = linear (±1 %)")
plt.tight_layout(); plt.show()

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("07", 6, "Dunkelstrom", tag="optional",
    intro="Steigung der mittleren Dunkelwerte gegen die Belichtungszeit mal Gain. Gilt nur bei stabil geregelter Temperatur.",
    blocks=[ui.note("info", "Der Hersteller nennt 4,4·10⁻⁵ e⁻/s/Pixel bei −20 °C. Das liegt unter der Nachweisgrenze dieser Methode, dann ist nur eine **Obergrenze** aussagekräftig.")]))

In [ ]:
GAIN = float(TAB["Gain_PTC"].mean())
if DARKS:
    d = {}
    for r in DARKS:
        d.setdefault(round(r["exptime"], 3), []).append(np.mean([_clipped_mean(r["rois"][p]) for p in PLANES]))
    dts = np.array(sorted(d)); dm = np.array([np.mean(d[t]) for t in dts])
    DARK = dark_current_e_per_s(dts, dm, GAIN)
    ui.say(f"Dunkelstrom ≈ {DARK:.4f} e⁻/s/Pixel (bei der Temperatur der Aufnahmen; Hersteller: 0,000044 bei −20 °C, 0,000996 bei 0 °C)")
else:
    DARK = None
    ui.say("keine Darks — Dunkelstrom nicht gemessen")

In [ ]:
from varphot import nbui as ui

ui.show(ui.step("07", 7, "Ergebnis, Vergleich mit dem Hersteller, Speichern",
    intro="Abweichungen von mehr als 10 % zum Herstellerwert werden markiert.",
    blocks=[ui.note("warn", "**Gespeichert wird nur im Modus `real`:** Kameraprofil in `PHOTOMETRIE_PY_DATA/` und der Abschnitt `camera` in `observer.json`.")]))

In [ ]:
if COMPRESSED:
    ui.say("⚠ Ergebnis unter Vorbehalt: RAW-Kompression vermutet (Schritt 3.2).")
RES = dict(gain_e_per_adu=GAIN, readout_noise_e=float(TAB["Ausleserauschen_e"].median()), full_well_e=full_well_e(GAIN, MAX_ADU),
           offset_adu=float(np.mean(list(offset.values()))), saturation_adu=MAX_ADU)
cmp = pd.DataFrame([
    dict(Größe="Gain [e⁻/ADU]", gemessen=RES["gain_e_per_adu"], Hersteller=VENDOR["gain"]),
    dict(Größe="Ausleserauschen [e⁻]", gemessen=RES["readout_noise_e"], Hersteller=VENDOR["read_noise"]),
    dict(Größe="Full Well [e⁻]", gemessen=RES["full_well_e"], Hersteller=VENDOR["full_well"]),
]).set_index("Größe")
cmp["Hersteller"] = pd.to_numeric(cmp["Hersteller"])                          # None → NaN (keine Herstellerangabe)
cmp["Abweichung_%"] = 100 * (cmp["gemessen"] / cmp["Hersteller"] - 1)
cmp["Urteil"] = np.where(cmp["Hersteller"].isna(), "keine Herstellerangabe", np.where(cmp["Abweichung_%"].abs() > 10, "⚠ > 10 %", "ok"))
ui.show(ui.table(cmp.round(2)))
if MODE == "demo":
    ui.say("\nWahrheit der Demo:", {k: TRUTH[k] for k in ("gain", "read_noise")},
          "→ gefunden: Gain %.2f, Ausleserauschen %.2f" % (RES["gain_e_per_adu"], RES["readout_noise_e"]))

In [ ]:
lin_text = (f"linear ±1 % von {LINRES['adu_min']:.0f} bis {LINRES['adu_max']:.0f} ADU ({SETTING})")
CAM = Camera(model=MODEL, cam_type="CMOS", gain_e_per_adu=round(RES["gain_e_per_adu"], 3), readout_noise_e=round(RES["readout_noise_e"], 2),
             dark_current_e_per_s=None if DARK is None else round(DARK, 5), dark_temp_c=None, linearity=lin_text, bayer_pattern=PATTERN,
             bit_depth=BITS_USED, saturation_adu=round(RES["saturation_adu"], 1), offset_adu=round(RES["offset_adu"], 1),
             full_well_e=round(RES["full_well_e"]), linear_min_adu=round(LINRES["adu_min"]), linear_max_adu=round(LINRES["adu_max"]), gain_setting=SETTING,
             source=f"gemessen (NB07, {datetime.date.today().isoformat()})")
if MODE == "real":
    obs, own_sites = load_config()
    obs.camera = CAM                                          # aktive Kamera/Einstellung
    obs.cameras = upsert_camera(obs.cameras, CAM)             # alle Profile (Modell + Einstellung)
    path = save_config(obs, own_sites)
    prof = data_dir() / f"camera_{slug(MODEL)}_{slug(SETTING)}.json"
    prof.write_text(json.dumps({"camera": CAM.__dict__, "je_ebene": TAB.round(4).to_dict(orient="index"),
                                "linearitaet": {k: LINRES[k] for k in ("a", "b", "t_min", "t_max", "adu_min", "adu_max")}},
                               indent=2, ensure_ascii=False, default=float) + "\n", encoding="utf-8")
    ui.say("gespeichert:", path, "und", prof)
else:
    ui.say("DEMO: nichts gespeichert. Profil wäre:", CAM)